# 🔍 AIOps — Session 10
# Automated Root Cause Analysis for Kubernetes Incidents

---

| | |
|---|---|
| **Course** | AIOps Engineering |
| **Lab** | Lab 10 |
| **Topics** | Event Correlation · Causal Analysis · Topology-Aware RCA |
| **Duration** | ~90 minutes |
| **Platform** | Kubernetes (kubectl must be available in this JupyterLab environment) |

---

## 🎯 Learning Objectives

By completing this lab you will be able to:

1. **Deploy** a realistic microservices topology on Kubernetes and inject realistic failure scenarios
2. **Correlate** multi-source events (metrics, logs, K8s events) using temporal and structural signals
3. **Apply** causal analysis to distinguish root causes from cascading downstream effects
4. **Implement** topology-aware RCA using service dependency graph traversal
5. **Score and rank** root cause candidates using a weighted evidence model
6. **Generate** automated incident reports with remediation playbooks

---

## 🗺️ Lab Architecture

We simulate a **ride-sharing platform** running on Kubernetes:

```
  [mobile-gateway] ──▶ [trip-service]   ──▶ [driver-service]  ──▶ [location-db]  ← ROOT CAUSE
                   ├──▶ [pricing-service] ──▶ [surge-engine]
                   └──▶ [payment-service] ──▶ [billing-db]
                                         └──▶ [fraud-detector]
```

The lab injects a **database OOM (Out-of-Memory) failure** in `location-db` that propagates upward through the dependency chain.

---

## 📋 Lab Phases

| Phase | Title | Key Concept |
|-------|-------|-------------|
| **0** | Environment Setup | Install libraries |
| **1** | Deploy Microservices on K8s | `kubectl apply` |
| **2** | Inject Failure Scenarios | Fault Injection |
| **3** | Collect Multi-Source Evidence | Metrics · Logs · Events |
| **4** | Event Correlation | Temporal + Structural |
| **5** | Causal Analysis | Granger · Propagation Trees |
| **6** | Topology-Aware RCA | Graph Traversal + Scoring |
| **7** | Automated Incident Report | RCA Output |
| **8** | Cleanup | `kubectl delete` |

> ▶️ **Run each cell in order using `Shift + Enter`**

---
## Phase 0 — Environment Setup

**Purpose:** Install all Python libraries used in this lab and verify `kubectl` connectivity to the cluster.

### Libraries we need

| Library | Purpose |
|---------|----------|
| `pandas`, `numpy` | Data manipulation and numerical computation |
| `networkx` | Build and traverse the service dependency graph |
| `scikit-learn` | Anomaly detection and feature analysis |
| `matplotlib`, `seaborn` | Visualisations and heatmaps |
| `scipy` | Statistical methods for causal tests |
| `tabulate` | Formatted console tables |

> ⏱️ Takes ~60 seconds on first run.

In [ ]:
# ── Install required libraries ─────────────────────────────────────────────────
import sys
!{sys.executable} -m pip install -q \
    pandas numpy matplotlib seaborn networkx \
    scikit-learn scipy plotly tabulate
print("✅ Libraries installed")

### Imports and global settings

All imports are collected here so every subsequent cell stays focused on its own logic.
`random.seed(42)` makes metric synthesis fully reproducible — you will see the same
numbers every time you run the lab from scratch.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
# We centralise all imports here so later cells stay focused on logic.
import subprocess, json, time, hashlib, random, textwrap, math
from datetime import datetime, timedelta
from collections import defaultdict, Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import networkx as nx
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from scipy import stats
from tabulate import tabulate

# ── Reproducibility ───────────────────────────────────────────────────────────
random.seed(42)
np.random.seed(42)

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#f8f9fa',
    'axes.facecolor':   '#ffffff',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.size':        10,
})

print("✅ Imports ready")

### Verify kubectl connectivity

`kubectl` must be reachable before we deploy anything.
If this step fails, check your `KUBECONFIG` environment variable or cluster credentials.

In [ ]:
# ── Verify kubectl access ──────────────────────────────────────────────────────
# This confirms your JupyterLab environment can talk to the Kubernetes cluster.
# RCA requires live K8s events and pod state — kubectl must be working.

def run(cmd, check=False):
    """Run a shell command, print output, return (stdout, returncode)."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0 and result.stderr.strip():
        print("STDERR:", result.stderr.strip())
    return result.stdout.strip(), result.returncode

print("=" * 55)
print("kubectl version (client):")
run("kubectl version --client --short 2>/dev/null || kubectl version --client")

print("\nCluster nodes:")
out, rc = run("kubectl get nodes")

if rc == 0:
    print("\n✅ kubectl is connected to the cluster — ready to proceed!")
else:
    print("\n❌ kubectl cannot reach the cluster.")
    print("   Check your KUBECONFIG or cluster credentials before continuing.")

---
## Phase 1 — Deploy the Ride-Sharing Platform on Kubernetes

**Purpose:** Create a realistic microservices topology we can inject failures into and run RCA on.

### What gets deployed

We create a dedicated namespace `aiops-lab10` and deploy **9 services** modelling a ride-sharing backend:

| Tier | Services |
|------|----------|
| Gateway | `mobile-gateway` |
| Backend | `trip-service`, `driver-service`, `pricing-service`, `surge-engine`, `payment-service`, `fraud-detector` |
| Database | `billing-db`, `location-db` ← **this is where the failure originates** |

Each pod is configured with:
- **Prometheus scrape annotations** (for metric collection simulation)
- **Liveness and readiness probes** (so K8s detects unhealthy pods)
- **Resource limits** (realistic constraints that allow OOM injection)

The `DEPENDENCY_GRAPH` dictionary defined in the next cell is the most important data
structure in the lab — it powers the topology-aware RCA in Phase 6.

> ⏱️ Allow 30–60 seconds for all pods to reach `Running` state.

In [ ]:
# ── Create the lab namespace ───────────────────────────────────────────────────
# All resources go into 'aiops-lab10' — easy to clean up at the end.

NAMESPACE = "aiops-lab10"

ns_yaml = f"""\
apiVersion: v1
kind: Namespace
metadata:
  name: {NAMESPACE}
  labels:
    lab: session10
    managed-by: aiops-lab
"""

with open("/tmp/lab10-ns.yaml", "w") as f:
    f.write(ns_yaml)

run(f"kubectl apply -f /tmp/lab10-ns.yaml")
print(f"\n✅ Namespace '{NAMESPACE}' ready")

### Service definitions and the dependency graph

`DEPENDENCY_GRAPH` maps each service to its **direct dependencies** (the downstream
services it calls). This directed graph is the backbone of topology-aware RCA:

- An edge `A → B` means *A calls B* (A depends on B)
- When B fails, A is a **victim**, not a cause
- `location-db` has no outgoing edges — it is a **leaf node** and the correct root cause

In [ ]:
# ── Service definitions and dependency graph ───────────────────────────────────
# SERVICES: (name, port, replicas, tier)
# DEPENDENCY_GRAPH: upstream service → [downstream dependencies]
# This graph is the backbone of our topology-aware RCA in Phase 6.

SERVICES = [
    ("mobile-gateway",  8080, 2, "gateway"),
    ("trip-service",    8081, 2, "backend"),
    ("driver-service",  8082, 2, "backend"),
    ("pricing-service", 8083, 1, "backend"),
    ("surge-engine",    8084, 1, "backend"),
    ("payment-service", 8085, 2, "backend"),
    ("fraud-detector",  8086, 1, "backend"),
    ("billing-db",      5432, 1, "database"),
    ("location-db",     5433, 1, "database"),
]

# Caller → callees  (who does this service depend on?)
DEPENDENCY_GRAPH = {
    "mobile-gateway":  ["trip-service", "pricing-service", "payment-service"],
    "trip-service":    ["driver-service"],
    "driver-service":  ["location-db"],
    "pricing-service": ["surge-engine"],
    "surge-engine":    [],
    "payment-service": ["billing-db", "fraud-detector"],
    "fraud-detector":  [],
    "billing-db":      [],
    "location-db":     [],   # ← ROOT CAUSE node
}

def build_manifest(name, port, replicas, tier):
    return f"""
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: {name}
  namespace: {NAMESPACE}
  labels:
    app: {name}
    tier: {tier}
spec:
  replicas: {replicas}
  selector:
    matchLabels:
      app: {name}
  template:
    metadata:
      labels:
        app: {name}
        tier: {tier}
      annotations:
        prometheus.io/scrape: "true"
        prometheus.io/port: "{port}"
        prometheus.io/path: "/metrics"
    spec:
      containers:
      - name: {name}
        image: nginx:alpine
        ports:
        - containerPort: 80
        resources:
          requests: {{cpu: "25m", memory: "32Mi"}}
          limits:   {{cpu: "100m", memory: "128Mi"}}
        livenessProbe:
          httpGet: {{path: "/", port: 80}}
          initialDelaySeconds: 5
          periodSeconds: 10
        readinessProbe:
          httpGet: {{path: "/", port: 80}}
          initialDelaySeconds: 3
          periodSeconds: 5
---
apiVersion: v1
kind: Service
metadata:
  name: {name}
  namespace: {NAMESPACE}
spec:
  selector:
    app: {name}
  ports:
  - port: {port}
    targetPort: 80
"""

manifest_all = "".join(build_manifest(*s) for s in SERVICES)
with open("/tmp/lab10-services.yaml", "w") as f:
    f.write(manifest_all)

run(f"kubectl apply -f /tmp/lab10-services.yaml")
print(f"\n✅ {len(SERVICES)} services applied")

### Wait for pods to become ready

We poll each deployment's rollout status before continuing.
All pods must reach `1/1 Ready` — injecting a failure into a pod that is not yet
running would produce misleading telemetry.

In [ ]:
# ── Wait for pods and verify deployment ───────────────────────────────────────
# We wait for the rollout to complete, then show pod health.
# All pods should be 1/1 Ready before we inject failures.

print("⏳ Waiting for deployments to roll out...")
for name, *_ in SERVICES:
    run(f"kubectl rollout status deployment/{name} -n {NAMESPACE} --timeout=90s")

print("\n" + "=" * 55)
print("Pod Status:")
run(f"kubectl get pods -n {NAMESPACE} -o wide")

print("\nServices:")
run(f"kubectl get svc -n {NAMESPACE}")

print("\n✅ All services deployed and healthy")

---
## Phase 2 — Inject Failure Scenarios 💥

### 📖 Concept: Fault Injection for RCA Testing

> **Root Cause Analysis** only makes sense in the context of a real failure. We need a scenario where:
> 1. A **single root cause** triggers at a specific time
> 2. Its effects **cascade** through the dependency chain with realistic delays
> 3. Multiple **symptom signals** (metrics, logs, K8s events) appear across services

### Failure Scenario: `location-db` OOM Kill

At `T=0`, `location-db` runs out of memory:
- **T+0s** — `location-db` pod is OOM-killed by K8s
- **T+8s** — `driver-service` starts getting `Connection refused` errors
- **T+15s** — `trip-service` sees increased latency, begins timeout errors
- **T+22s** — `mobile-gateway` request error rate crosses 5% threshold
- **T+28s** — Alerts fire across the entire service chain

This is a **classic upstream database failure cascade** — exactly the scenario where naive alert-driven triage would lead engineers to the wrong services first.

In [ ]:
# ── Inject the OOM failure into location-db ────────────────────────────────────
# We patch the deployment to set memory limit = memory request (forces OOM).
# This is a real K8s operation — the pod will be killed and begin CrashLoopBackOff.

T0 = datetime.now()
print(f"⚡ Injecting OOM failure into 'location-db' at T0 = {T0.strftime('%H:%M:%S')}")

oom_patch = json.dumps({
    "spec": {
        "template": {
            "spec": {
                "containers": [{
                    "name": "location-db",
                    "resources": {
                        "requests": {"memory": "4Mi"},
                        "limits":   {"memory": "4Mi"}  # too small → OOM
                    }
                }]
            }
        }
    }
})

run(f"kubectl patch deployment location-db -n {NAMESPACE} --type=merge -p '{oom_patch}'")

print("\n⏳ Waiting 20 seconds for failure to propagate...")
time.sleep(20)

print("\nCurrent pod states (look for OOMKilled / CrashLoopBackOff):")
run(f"kubectl get pods -n {NAMESPACE}")

print("\n✅ Failure injected. Now we collect evidence.")

---
## Phase 3 — Collect Multi-Source Evidence 📡

### 📖 Concept: Why Multi-Source Evidence Matters

A single data source is almost never enough to identify a root cause. RCA systems must **fuse evidence from multiple telemetry streams**:

| Source | What it tells us | Limitation |
|--------|-----------------|------------|
| **Metrics** | Quantitative signal (latency, errors, CPU) | High noise, hard to interpret alone |
| **Logs** | Rich context, error messages | Unstructured, high volume |
| **K8s Events** | Pod lifecycle, OOMKill, restarts | Ground truth for infra failures |
| **Traces** | Request path across services | Not always available |

In this phase we:
1. Collect real K8s events from the cluster
2. Synthesise realistic metrics (error rate, latency, CPU) for each service
3. Synthesise application log anomalies

The synthesis reflects the propagation delay model from Phase 2.

### Step 3a — Collect K8s Events

Kubernetes records infrastructure lifecycle events in the cluster's event log.
These are the **highest-fidelity signals** for root cause analysis:
- `OOMKilling` → container ran out of memory (our injected fault)
- `CrashLoopBackOff` → container keeps restarting after failure
- `Unhealthy` → liveness / readiness probe is failing
- `BackOff` → K8s is backing off restart attempts

We fetch real events from the cluster and supplement them with a set of
**simulated events** that model the exact cascade we would see in production.

In [ ]:
# ── Collect real K8s events from the cluster ───────────────────────────────────
# K8s events are the ground truth for infrastructure failures.
# They record OOMKill, CrashLoopBackOff, pod restarts, liveness failures, etc.
# We parse them into a structured DataFrame for later analysis.

print("Collecting K8s events from cluster...")
events_raw, rc = run(
    f"kubectl get events -n {NAMESPACE} "
    f"--sort-by=.lastTimestamp "
    f"-o json 2>/dev/null"
)

k8s_events = []
if rc == 0 and events_raw.strip():
    try:
        ev_data = json.loads(events_raw)
        for item in ev_data.get("items", []):
            k8s_events.append({
                "time":    item.get("lastTimestamp", ""),
                "type":    item.get("type", ""),
                "reason":  item.get("reason", ""),
                "object":  item.get("involvedObject", {}).get("name", ""),
                "kind":    item.get("involvedObject", {}).get("kind", ""),
                "message": item.get("message", ""),
            })
    except json.JSONDecodeError:
        print("⚠️  Could not parse events JSON — using simulated events.")

# ── Always add simulated events to demonstrate the full RCA pipeline ───────────
# These represent the events we WOULD see in production after an OOM event.
simulated_k8s = [
    {"time": (T0 + timedelta(seconds=2)).isoformat(),  "type": "Warning", "reason": "OOMKilling",       "object": "location-db-7d4b9f",  "kind": "Pod",        "message": "Memory limit exceeded: killed process 1 (postgres)"},
    {"time": (T0 + timedelta(seconds=4)).isoformat(),  "type": "Warning", "reason": "BackOff",          "object": "location-db-7d4b9f",  "kind": "Pod",        "message": "Back-off restarting failed container location-db"},
    {"time": (T0 + timedelta(seconds=8)).isoformat(),  "type": "Warning", "reason": "Unhealthy",        "object": "driver-service-2c4a1", "kind": "Pod",        "message": "Readiness probe failed: connection refused to location-db:5433"},
    {"time": (T0 + timedelta(seconds=9)).isoformat(),  "type": "Warning", "reason": "FailedMount",      "object": "location-db-7d4b9f",  "kind": "Pod",        "message": "Unable to attach volume: timeout"},
    {"time": (T0 + timedelta(seconds=15)).isoformat(), "type": "Warning", "reason": "Unhealthy",        "object": "trip-service-9f3c2",  "kind": "Pod",        "message": "Liveness probe failed: HTTP 503 from /health"},
    {"time": (T0 + timedelta(seconds=18)).isoformat(), "type": "Normal",  "reason": "ScalingReplicaSet","object": "trip-service",         "kind": "Deployment", "message": "Scaled down replica set trip-service to 1"},
    {"time": (T0 + timedelta(seconds=22)).isoformat(), "type": "Warning", "reason": "Unhealthy",        "object": "mobile-gateway-1a7b3", "kind": "Pod",        "message": "Readiness probe failed: downstream dependency unavailable"},
    {"time": (T0 + timedelta(seconds=28)).isoformat(), "type": "Warning", "reason": "CrashLoopBackOff", "object": "location-db-7d4b9f",  "kind": "Pod",        "message": "Container location-db is in CrashLoopBackOff"},
]
k8s_events.extend(simulated_k8s)

df_k8s = pd.DataFrame(k8s_events)
print(f"\n✅ Collected {len(df_k8s)} K8s events")
print(tabulate(df_k8s[["reason", "object", "kind", "message"]].head(10),
               headers="keys", tablefmt="rounded_outline", showindex=False))

### Step 3b — Synthesise Service Metrics

In production, metrics stream from **Prometheus / Datadog / Dynatrace**.
Here we synthesise them using a **sigmoid ramp function** that matches the
propagation delay model: each service starts degrading only *after* its dependency fails.

Key metric columns:
- `error_rate` — HTTP 5xx rate (0.0 – 1.0), the primary RCA signal
- `latency_p99` — 99th-percentile response time in milliseconds
- `cpu_util` — CPU utilisation (0.0 – 1.0)
- `restart_count` — Pod restart count detected from K8s

In [ ]:
# ── Synthesise realistic metrics for each service ─────────────────────────────
# In production these would come from Prometheus / Datadog / Dynatrace.
# We synthesise them here with realistic propagation delays from T0.
#
# Each service has:
#   - error_rate  : HTTP 5xx rate (0–1)
#   - latency_p99 : 99th percentile latency in ms
#   - cpu_util    : CPU utilisation (0–1)
#   - restart_count: pod restart count (from K8s)
#
# The propagation model: failure starts at location-db (T+0)
# and spreads upstream with realistic delays.

TIMESTAMPS = [T0 + timedelta(seconds=i*5) for i in range(30)]  # 2.5 min of telemetry
T_SECONDS = [(t - T0).total_seconds() for t in TIMESTAMPS]

# Propagation delay per service (seconds after T0 before impact is visible)
IMPACT_DELAY = {
    "location-db":     0,
    "driver-service":  8,
    "trip-service":    15,
    "pricing-service": 40,   # not affected by location-db
    "surge-engine":    50,   # not affected
    "mobile-gateway":  22,
    "payment-service": 45,   # not affected
    "fraud-detector":  50,   # not affected
    "billing-db":      55,   # not affected
}

# Severity of impact (max error rate at peak)
PEAK_ERROR = {
    "location-db":     0.98,
    "driver-service":  0.80,
    "trip-service":    0.65,
    "mobile-gateway":  0.40,
    "pricing-service": 0.02,  # not on critical path
    "surge-engine":    0.02,
    "payment-service": 0.03,
    "fraud-detector":  0.02,
    "billing-db":      0.01,
}

def sigmoid_ramp(t_seconds, delay, peak, ramp_rate=0.2):
    """Ramp up error rate after delay using sigmoid for realism."""
    shifted = t_seconds - delay
    return peak / (1 + np.exp(-ramp_rate * shifted))

metrics_rows = []
for svc, delay in IMPACT_DELAY.items():
    for ts, t_sec in zip(TIMESTAMPS, T_SECONDS):
        err = sigmoid_ramp(t_sec, delay, PEAK_ERROR[svc])
        err = min(err + random.gauss(0, 0.02), 1.0)
        lat_base = 50 if "db" in svc else 120
        lat = lat_base + err * 2000 + random.gauss(0, 20)
        cpu = 0.2 + err * 0.5 + random.gauss(0, 0.05)
        metrics_rows.append({
            "timestamp":     ts,
            "service":       svc,
            "error_rate":    round(max(0, err), 4),
            "latency_p99":   round(max(5, lat), 1),
            "cpu_util":      round(min(1.0, max(0, cpu)), 3),
            "restart_count": int(err > 0.5) * random.randint(1, 5),
        })

df_metrics = pd.DataFrame(metrics_rows)
print(f"✅ Generated {len(df_metrics)} metric data points across {df_metrics['service'].nunique()} services")
print("\nPeak error rates per service:")
peak = df_metrics.groupby("service")["error_rate"].max().sort_values(ascending=False)
for svc, rate in peak.items():
    bar = "█" * int(rate * 20)
    print(f"  {svc:<22} {bar:<20} {rate:.2%}")

### Step 3c — Synthesise Application Logs

Application logs contain **semantic clues** about the failure mode.
Notice how the error messages follow the cascade chain:
- `location-db` → `FATAL: out of memory`
- `driver-service` → `connection refused to location-db:5433`
- `trip-service` → `circuit breaker OPEN for driver-service`
- `mobile-gateway` → `trip-service health check failed`

This *message trail* is a critical signal for semantic log correlation in Phase 4.

In [ ]:
# ── Synthesise application log anomalies ──────────────────────────────────────
# In production these would come from Elasticsearch / Loki / Splunk.
# We generate realistic log lines that would appear during a DB outage.
# Key: log messages contain clues about the failure — RCA uses these.

LOG_TEMPLATES = {
    "location-db": [
        "FATAL: out of memory",
        "ERROR: could not write to file 'pg_wal': No space left on device",
        "PANIC: could not write to file: No space left",
        "LOG: database system is shut down",
    ],
    "driver-service": [
        "ERROR: dial tcp location-db:5433: connect: connection refused",
        "WARN: failed to get driver location after 3 retries",
        "ERROR: context deadline exceeded (timeout=5s) calling location-db",
        "ERROR: pq: could not connect to server: Connection refused",
    ],
    "trip-service": [
        "ERROR: downstream service driver-service returned 503",
        "WARN: circuit breaker OPEN for driver-service after 5 failures",
        "ERROR: trip creation failed: driver assignment unavailable",
    ],
    "mobile-gateway": [
        "WARN: trip-service health check failed (HTTP 503)",
        "ERROR: request to trip-service timed out after 10s",
        "WARN: fallback triggered for trip endpoint",
    ],
    "payment-service": [
        "INFO: request processed successfully",
        "INFO: payment authorised",
    ],
    "pricing-service": ["INFO: price calculated"],
    "surge-engine":    ["INFO: surge multiplier updated"],
    "fraud-detector":  ["INFO: transaction cleared"],
    "billing-db":      ["INFO: checkpoint complete"],
}

log_rows = []
for svc, templates in LOG_TEMPLATES.items():
    delay = IMPACT_DELAY[svc]
    peak = PEAK_ERROR[svc]
    n_logs = int(peak * 100) + random.randint(5, 15)
    for _ in range(n_logs):
        t_offset = delay + random.expovariate(0.05)
        ts = T0 + timedelta(seconds=min(t_offset, 140))
        msg = random.choice(templates)
        level = "ERROR" if "ERROR" in msg or "FATAL" in msg or "PANIC" in msg else (
                "WARN"  if "WARN"  in msg else "INFO")
        log_rows.append({"timestamp": ts, "service": svc, "level": level, "message": msg})

df_logs = pd.DataFrame(log_rows).sort_values("timestamp").reset_index(drop=True)

# Summary
log_summary = df_logs.groupby(["service", "level"]).size().unstack(fill_value=0)
print(f"✅ Generated {len(df_logs)} log lines")
print(tabulate(log_summary, headers="keys", tablefmt="rounded_outline"))

print("\nSample ERROR logs:")
sample = df_logs[df_logs["level"].isin(["ERROR","WARN"])].head(6)
for _, row in sample.iterrows():
    print(f"  [{row.service}] {row.message}")

---
## Phase 4 — Event Correlation 🔗

### 📖 Concept: Event Correlation

> **Event correlation** is the process of identifying relationships between individual telemetry signals to understand *which events belong to the same incident*.

We apply three complementary techniques:

| Technique | Principle | Strength |
|-----------|-----------|----------|
| **Temporal Correlation** | Events close in time likely share a cause | Works across all services |
| **Structural Correlation** | Events in dependent services likely related | Uses topology knowledge |
| **Semantic Correlation** | Similar error messages cluster together | Works on logs |

The output of correlation is a set of **correlated event groups** — clusters of signals that all belong to the same incident.

### Technique 4a — Temporal Correlation

**Idea:** If two services both develop anomalies within the same 30-second window,
they are almost certainly part of the same incident.

**Algorithm:**
1. Find the timestamp when each service's error rate first exceeds 10%  (`T_first_anomaly`)
2. Sort services by `T_first_anomaly`
3. Apply a sliding 30-second window to group services whose onset times are close together
4. Services in the same window = **temporally correlated group**

In [ ]:
# ── Technique 4a: Temporal Correlation ────────────────────────────────────────
# Find which services showed anomalies within a sliding time window.
# We use a 30-second window: if two services both spike in the same window,
# they are likely part of the same incident.
#
# Method:
#   1. Detect when each service first crosses the anomaly threshold (err > 0.1)
#   2. Cluster services whose first-anomaly timestamps are within 30s of each other
#   3. Services in the same cluster are correlated

ANOMALY_THRESHOLD = 0.10  # error rate > 10% = anomalous
CORRELATION_WINDOW = 30   # seconds

# Find T_first_anomaly for each service
first_anomaly = {}
for svc in df_metrics["service"].unique():
    svc_data = df_metrics[df_metrics["service"] == svc].sort_values("timestamp")
    anomalous = svc_data[svc_data["error_rate"] > ANOMALY_THRESHOLD]
    if not anomalous.empty:
        t_first = anomalous.iloc[0]["timestamp"]
        delay_s = (t_first - T0).total_seconds()
        first_anomaly[svc] = {"t_first": t_first, "delay_s": delay_s,
                               "peak_err": svc_data["error_rate"].max()}

df_anomaly = pd.DataFrame(first_anomaly).T.sort_values("delay_s")
print("⏱️  Time-to-first-anomaly per service:")
print(tabulate(df_anomaly[["delay_s", "peak_err"]].rename(
    columns={"delay_s": "Onset (s after T0)", "peak_err": "Peak Error Rate"}),
    headers="keys", tablefmt="rounded_outline", floatfmt=(".1f", ".2%")))

# Cluster into correlated groups using sliding window
sorted_svcs = df_anomaly.sort_values("delay_s").index.tolist()
corr_groups = []
visited = set()

for i, svc_a in enumerate(sorted_svcs):
    if svc_a in visited:
        continue
    group = [svc_a]
    visited.add(svc_a)
    t_a = df_anomaly.loc[svc_a, "delay_s"]
    for svc_b in sorted_svcs[i+1:]:
        if svc_b not in visited:
            t_b = df_anomaly.loc[svc_b, "delay_s"]
            if abs(t_b - t_a) <= CORRELATION_WINDOW:
                group.append(svc_b)
                visited.add(svc_b)
    corr_groups.append(group)

print(f"\n📦 Temporal correlation groups ({CORRELATION_WINDOW}s window):")
for i, grp in enumerate(corr_groups):
    print(f"  Group {i+1}: {', '.join(grp)}")

### Technique 4b — Structural (Topology) Correlation

**Idea:** If two services are both showing anomalies AND are connected in the
dependency graph, they almost certainly belong to the same incident.

**Algorithm:**
1. Build a directed graph from `DEPENDENCY_GRAPH`
2. Extract the **subgraph** containing only the anomalous services
3. Find **weakly connected components** of that subgraph
4. Services in the same component = **structurally correlated**

We also compute the **Jaccard agreement** between temporal and structural groupings.
High agreement confirms the correlation is real, not coincidental.

In [ ]:
# ── Technique 4b: Structural (Topology) Correlation ───────────────────────────
# Using the dependency graph, find which affected services are topologically
# connected. Connected affected services are almost certainly in the same incident.
#
# Method:
#   1. Build a directed graph from DEPENDENCY_GRAPH
#   2. Find the set of affected services (from first_anomaly)
#   3. Find connected components among affected services
#   4. Services in the same connected component = structural correlation

G = nx.DiGraph()
for caller, callees in DEPENDENCY_GRAPH.items():
    G.add_node(caller)
    for callee in callees:
        G.add_edge(caller, callee)

affected_services = set(first_anomaly.keys())

# Subgraph of only affected services
G_affected = G.subgraph(affected_services).copy()

# Find weakly connected components
wcc = list(nx.weakly_connected_components(G_affected))

print("🕸️  Structural correlation (dependency subgraph of affected services):")
for i, comp in enumerate(wcc):
    print(f"  Component {i+1}: {', '.join(sorted(comp))}")

# Services NOT topologically connected to the main incident
unaffected = set(G.nodes()) - affected_services
if unaffected:
    print(f"\n  ✅ Not in incident chain: {', '.join(sorted(unaffected))}")

# Agreement check: temporal vs structural correlation
largest_struct = sorted(wcc, key=len, reverse=True)[0]
largest_temp   = sorted(corr_groups, key=len, reverse=True)[0]
agreement = len(set(largest_struct) & set(largest_temp)) / len(set(largest_struct) | set(largest_temp))
print(f"\n📊 Temporal ∩ Structural agreement (Jaccard): {agreement:.0%}")
INCIDENT_SERVICES = set(largest_struct) | set(largest_temp)
print(f"   Final correlated incident services: {sorted(INCIDENT_SERVICES)}")

### Technique 4c — Semantic Log Correlation

**Idea:** Log messages that share the same error keyword across multiple services
point to a shared underlying cause.

**Algorithm:**
1. Filter to ERROR/WARN logs from services in the incident scope
2. Classify each log line into a semantic category (`OOM`, `DB_CONN`, `TIMEOUT`, etc.)
3. Count category occurrences per service
4. Services sharing the same dominant category are **semantically correlated**

> 💡 Notice how `DB_CONN` errors appear in both `driver-service` and `location-db`
> — this is the explicit link between victim and cause in the log data.

In [ ]:
# ── Technique 4c: Semantic Log Correlation ────────────────────────────────────
# Cluster log messages by keyword similarity to find recurring error themes.
# Common themes across multiple services indicate a shared root cause.
#
# Method:
#   1. Extract ERROR/WARN logs from incident services
#   2. Tokenise and find the most frequent error keywords
#   3. Group log lines by their dominant keyword cluster

ERROR_KEYWORDS = {
    "DB_CONN":     ["connection refused", "connect:", "pq:", "dial tcp"],
    "TIMEOUT":     ["timeout", "deadline exceeded", "timed out"],
    "OOM":         ["out of memory", "oom", "memory"],
    "UNAVAILABLE": ["503", "unavailable", "circuit breaker"],
    "CRASH":       ["fatal", "panic", "shut down", "crashloop"],
}

def classify_log(message):
    msg_lower = message.lower()
    for category, keywords in ERROR_KEYWORDS.items():
        if any(kw in msg_lower for kw in keywords):
            return category
    return "OTHER"

df_err_logs = df_logs[
    (df_logs["level"].isin(["ERROR", "WARN"])) &
    (df_logs["service"].isin(INCIDENT_SERVICES))
].copy()

df_err_logs["category"] = df_err_logs["message"].apply(classify_log)

pivot = df_err_logs.groupby(["service", "category"]).size().unstack(fill_value=0)
print("📋 Log semantic clusters per service:")
print(tabulate(pivot, headers="keys", tablefmt="rounded_outline"))

print("\n💡 Insight:")
dominant = df_err_logs["category"].value_counts()
for cat, count in dominant.head(3).items():
    svcs = df_err_logs[df_err_logs["category"] == cat]["service"].unique()
    print(f"  [{cat}] appeared {count}x across: {', '.join(svcs)}")

### Visualise — Error Rate Propagation Timeline

This chart is the clearest visual proof of the cascade.
The key observation: **`location-db`'s curve (red, bold) rises first**,
and each dependent service follows in topological order.

Without the dependency graph, an engineer looking at this chart might
focus on `mobile-gateway` — it's large and user-facing.
Phase 6 will show how topology knowledge corrects that misdiagnosis.

In [ ]:
# ── Visualise: Error Rate Propagation Timeline ─────────────────────────────────
# This chart shows HOW the failure propagated through the service mesh over time.
# The left-to-right ordering of the curves reveals the propagation direction:
# the service whose curve rises FIRST is the likely root cause.

COLORS = {
    "location-db":    "#d62728",
    "driver-service": "#ff7f0e",
    "trip-service":   "#e377c2",
    "mobile-gateway": "#9467bd",
    "pricing-service":"#2ca02c",
    "payment-service":"#17becf",
    "surge-engine":   "#bcbd22",
    "fraud-detector": "#8c564b",
    "billing-db":     "#aec7e8",
}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

for svc in df_metrics["service"].unique():
    data = df_metrics[df_metrics["service"] == svc].sort_values("timestamp")
    x = [(t - T0).total_seconds() for t in data["timestamp"]]
    ax1.plot(x, data["error_rate"], label=svc, color=COLORS.get(svc, "gray"),
             linewidth=2.5 if svc == "location-db" else 1.2)
    ax2.plot(x, data["latency_p99"], color=COLORS.get(svc, "gray"),
             linewidth=2.5 if svc == "location-db" else 1.2)

ax1.axhline(ANOMALY_THRESHOLD, color="red", linestyle="--", alpha=0.5, label="Anomaly threshold")
ax1.set_ylabel("Error Rate")
ax1.set_title("🔴 Error Rate Propagation — Notice location-db rises FIRST", fontweight="bold")
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax1.legend(ncol=3, fontsize=8, loc="upper right")

ax2.set_ylabel("P99 Latency (ms)")
ax2.set_xlabel("Seconds after T0 (failure injection)")
ax2.set_title("⏱️  P99 Latency — Cascading increase across dependent services")

# Mark T0
for ax in [ax1, ax2]:
    ax.axvline(0, color="black", linestyle=":", alpha=0.7)
    ax.text(1, ax.get_ylim()[1] * 0.9, "T0 (OOM)", fontsize=8, color="black")

plt.tight_layout()
plt.savefig("/tmp/lab10_propagation.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Propagation timeline chart rendered")

---
## Phase 5 — Causal Analysis 🔬

### 📖 Concept: Correlation ≠ Causation

> Two services failing at the same time doesn't tell us which one *caused* the other to fail. We need **causal analysis** to determine direction.

We use three causal methods:

1. **Temporal Precedence** — "A causes B" requires A to change *before* B (necessary but not sufficient)
2. **Propagation Tree** — Starting from the earliest anomaly, build a tree showing how impact spread downstream
3. **Anomaly Score Decay** — Root causes have the *highest* anomaly magnitude; downstream effects are weaker and later

Together, these methods let us assign a **causal confidence score** to each service.

### Method 5a — Temporal Precedence Analysis

**Principle:** If A's anomaly starts *before* B's anomaly, A could be a cause of B.
This is a necessary (but not sufficient) condition for causation.

We build a **precedence matrix**: entry `[A, B] = ✓` means A's onset
preceded B's by at least 3 seconds. The service with the most `✓` entries in
its row is the earliest in the causal chain — a strong root cause indicator.

In [ ]:
# ── Causal Method 1: Temporal Precedence Analysis ─────────────────────────────
# If service A's anomaly onset precedes service B's onset AND
# A is an upstream dependency of B, then A likely caused B's failure.
#
# We compute a directed "causation matrix" where entry [A,B] = 1 if
# A is a plausible cause of B (A precedes B AND A depends on B).

# Only consider incident services that actually had anomalies
affected = sorted(first_anomaly.keys(), key=lambda s: first_anomaly[s]["delay_s"])

print("⏱️  Temporal Precedence Matrix (row → might cause column):")
print("   A 'YES' means row-service anomaly PRECEDED column-service anomaly")
print()

# Build matrix
precedence = {}
for a in affected:
    precedence[a] = {}
    for b in affected:
        if a == b:
            precedence[a][b] = "-"
        else:
            ta = first_anomaly[a]["delay_s"]
            tb = first_anomaly[b]["delay_s"]
            if ta < tb - 3:   # A preceded B by at least 3s
                precedence[a][b] = "✓"
            else:
                precedence[a][b] = " "

df_prec = pd.DataFrame(precedence).T
print(tabulate(df_prec, headers="keys", tablefmt="rounded_outline"))

# Count how many services each service PRECEDED (higher = earlier in causal chain)
preceded_count = {
    svc: sum(1 for v in precedence[svc].values() if v == "✓")
    for svc in affected
}

print("\n🏆 Temporal precedence score (higher = earlier in chain):")
for svc, score in sorted(preceded_count.items(), key=lambda x: -x[1]):
    bar = "▰" * score
    print(f"  {svc:<22} {bar} ({score})")

### Method 5b — Failure Propagation Tree

**Principle:** Starting from the earliest-failing service, we trace how failure
*spread* through the dependency graph.

If service A failed at `T+0s` and its dependent B failed at `T+8s`, we draw
a directed edge `A → B` with `propagation_delay = 8s`.

The **root** of the resulting tree (the node with no incoming edges)
is the strongest root cause candidate from this method.

In [ ]:
# ── Causal Method 2: Failure Propagation Tree ──────────────────────────────────
# Starting from the earliest anomaly service, we build a directed tree
# showing how failure propagated through the dependency graph.
#
# Algorithm:
#   1. Start from the service with the earliest anomaly onset (T_min)
#   2. For each downstream dependent, check if its onset is > T_min + propagation_lag
#   3. If yes → add directed edge from cause to effect
#   4. The root of the tree = root cause candidate

PROPAGATION_LAG_TOLERANCE = 5  # seconds: allow ±5s for realistic lag

def build_propagation_tree(first_anomaly_map, dep_graph):
    """Build a directed propagation tree from anomaly timing."""
    tree = nx.DiGraph()
    for svc in first_anomaly_map:
        tree.add_node(svc, onset=first_anomaly_map[svc]["delay_s"],
                       peak_err=first_anomaly_map[svc]["peak_err"])

    for caller, callees in dep_graph.items():
        if caller not in first_anomaly_map:
            continue
        t_caller = first_anomaly_map[caller]["delay_s"]
        for callee in callees:
            if callee not in first_anomaly_map:
                continue
            t_callee = first_anomaly_map[callee]["delay_s"]
            # The callee (upstream service) fails BEFORE the caller (downstream)
            # because callee is what the caller depends on
            if t_callee < t_caller - PROPAGATION_LAG_TOLERANCE:
                # callee caused caller's failure
                tree.add_edge(callee, caller,
                               propagation_delay=round(t_caller - t_callee, 1))
    return tree

prop_tree = build_propagation_tree(first_anomaly, DEPENDENCY_GRAPH)

# Identify root of the tree (nodes with no incoming edges = no one caused THEM)
roots = [n for n in prop_tree.nodes() if prop_tree.in_degree(n) == 0]

print("🌳 Failure Propagation Tree:")
for root in roots:
    print(f"  ROOT: {root}  (onset T+{first_anomaly.get(root, {}).get('delay_s', '?'):.0f}s)")
    for edge in nx.bfs_edges(prop_tree, root):
        depth = nx.shortest_path_length(prop_tree, root, edge[1])
        indent = "  " + "  │ " * (depth - 1) + "  └─▶ "
        delay = prop_tree[edge[0]][edge[1]].get("propagation_delay", "?")
        t1 = first_anomaly.get(edge[1], {}).get("delay_s", "?")
        print(f"{indent}{edge[1]}  (+{delay}s propagation, onset T+{t1:.0f}s)")

print(f"\n✅ Root cause candidate(s) from propagation tree: {roots}")

### Method 5c — Anomaly Magnitude + K8s Event Scoring

**Principle:** Root causes exhibit the most severe anomaly AND generate the
most serious K8s infrastructure events.

We combine four signals into a weighted **composite causal score**:

| Signal | Weight | Rationale |
|--------|--------|-----------|
| Peak error rate | 35% | Higher severity = more likely cause |
| Onset timing (inverted) | 30% | Earlier = more likely cause |
| K8s event severity | 20% | OOMKill > CrashLoop > Unhealthy |
| Temporal precedence count | 15% | From Method 5a |

Formula: `0.35 × peak_err + 0.30 × onset_score + 0.20 × k8s_score + 0.15 × prec_score`

In [ ]:
# ── Causal Method 3: Anomaly Magnitude & K8s Event Scoring ────────────────────
# Root causes exhibit:
#   (a) The HIGHEST anomaly magnitude (peak error rate)
#   (b) The MOST severe K8s events (OOMKilled > CrashLoop > Unhealthy)
#   (c) The EARLIEST onset
#
# We compute a composite anomaly score for each service.

K8S_EVENT_WEIGHTS = {
    "OOMKilling":        10,
    "CrashLoopBackOff":  8,
    "BackOff":           6,
    "Unhealthy":         5,
    "FailedMount":       4,
    "ScalingReplicaSet": 2,
    "Killing":           3,
}

# Compute K8s event score per service
k8s_scores = defaultdict(float)
for _, row in df_k8s.iterrows():
    obj = row.get("object", "")
    reason = row.get("reason", "")
    for svc_name, _, _, _ in SERVICES:
        if svc_name in obj:
            k8s_scores[svc_name] += K8S_EVENT_WEIGHTS.get(reason, 1)

# Compute composite score
causal_scores = {}
for svc in affected:
    onset    = first_anomaly[svc]["delay_s"]
    peak_err = first_anomaly[svc]["peak_err"]
    k8s      = k8s_scores.get(svc, 0)
    prec     = preceded_count.get(svc, 0)

    # Earlier onset → higher score (invert delay: max_delay - delay)
    max_delay = max(v["delay_s"] for v in first_anomaly.values())
    onset_score = (max_delay - onset) / max_delay if max_delay > 0 else 0

    # Weighted composite
    score = (0.35 * peak_err +
             0.30 * onset_score +
             0.20 * min(k8s / 20, 1.0) +
             0.15 * min(prec / len(affected), 1.0))

    causal_scores[svc] = {
        "peak_err":    round(peak_err, 3),
        "onset_s":     round(onset, 1),
        "k8s_weight":  round(k8s, 1),
        "prec_score":  prec,
        "CAUSAL_SCORE": round(score, 4),
    }

df_causal = pd.DataFrame(causal_scores).T.sort_values("CAUSAL_SCORE", ascending=False)

print("📊 Causal Analysis — Composite Scores per Service:")
print(tabulate(df_causal, headers="keys", tablefmt="rounded_outline",
               floatfmt=(".3f", ".1f", ".1f", ".0f", ".4f")))

TOP_CAUSE = df_causal.index[0]
print(f"\n🏆 Top causal candidate: {TOP_CAUSE}  (score: {df_causal.iloc[0]['CAUSAL_SCORE']:.4f})")

---
## Phase 6 — Topology-Aware RCA 🗺️

### 📖 Concept: Why Topology Changes Everything

> Without topology knowledge, a pure metrics-based system might flag `mobile-gateway` as the root cause because it serves user traffic. **But mobile-gateway is a victim, not the cause.**

Topology-aware RCA uses the service dependency graph to:
1. **Re-weight** causal scores — deeper nodes (closer to data layer) get a topology bonus
2. **Prune false positives** — services with no upstream dependency on the flagged anomalous services are excluded
3. **Calculate blast radius** — how many services would recover if we fix each candidate

The final RCA output ranks root cause candidates with a combined **topology-adjusted confidence score**.

### Build the annotated dependency graph and rank candidates

We annotate each node with causal evidence from Phase 5, then compute
the topology-adjusted score using two additional signals:

- **Depth bonus** — `location-db` is at depth 3 from the gateway; `mobile-gateway`
  is at depth 0. Deeper leaf nodes that fail are more likely to be the root cause.
- **Blast radius** — the count of services that would recover if we fix this one.
  A true root cause has the maximum blast radius.

**Bug fixed:** The original code used `successors & affected` where `affected` was
a `list`. Python's `&` set-intersection operator requires both sides to be `set`.
Fixed by converting: `set(ancestors) & set(affected)`.

**Also fixed:** `df_rca` now keeps `service` as a regular column (using
`reset_index(drop=True)` instead of `set_index('rank')`) so all downstream
cells that reference `df_rca['service']` work correctly.

In [ ]:
# Topology-aware RCA: annotate the dependency graph and compute topology-adjusted scores

# Compute depth of each node from mobile-gateway (0 = gateway, max = leaf/database)
depths    = nx.single_source_shortest_path_length(G, "mobile-gateway")
max_depth = max(depths.values())

# Annotate graph nodes with evidence from causal analysis
for node in G.nodes():
    G.nodes[node]["depth"] = depths.get(node, 0)
    if node in causal_scores:
        G.nodes[node]["causal_score"] = causal_scores[node]["CAUSAL_SCORE"]
        G.nodes[node]["peak_err"]     = causal_scores[node]["peak_err"]
        G.nodes[node]["onset_s"]      = causal_scores[node]["onset_s"]
    else:
        G.nodes[node]["causal_score"] = 0.0
        G.nodes[node]["peak_err"]     = 0.0
        G.nodes[node]["onset_s"]      = 9999.0

# Convert affected list to set for set-intersection operations
affected_set = set(affected)

rca_candidates = []
for svc in affected:
    depth      = G.nodes[svc]["depth"]
    causal     = G.nodes[svc]["causal_score"]
    topo_bonus = depth / max_depth if max_depth > 0 else 0

    # Blast radius: how many affected services would recover if we fix this one?
    # nx.ancestors(G, svc) returns every node that has svc as a downstream dependency
    ancestors    = set(nx.ancestors(G, svc))
    blast_radius = len(ancestors & affected_set)  # FIX: both operands are now sets

    topo_score = 0.60 * causal + 0.25 * topo_bonus + 0.15 * min(blast_radius / 5, 1.0)

    rca_candidates.append({
        "service":        svc,
        "depth":          depth,
        "causal_score":   round(causal, 4),
        "topo_bonus":     round(topo_bonus, 3),
        "blast_radius":   blast_radius,
        "TOPO_RCA_SCORE": round(topo_score, 4),
    })

# FIX: keep service as a regular column so downstream cells can access df_rca["service"]
df_rca = (pd.DataFrame(rca_candidates)
            .sort_values("TOPO_RCA_SCORE", ascending=False)
            .reset_index(drop=True))
df_rca.index     = df_rca.index + 1
df_rca.index.name = "rank"

print("🎯 Topology-Aware RCA Rankings:")
print(tabulate(df_rca, headers="keys", tablefmt="rounded_outline"))

ROOT_CAUSE = df_rca.iloc[0]["service"]
RCA_SCORE  = df_rca.iloc[0]["TOPO_RCA_SCORE"]
print(f"\n✅ RCA VERDICT: ROOT CAUSE = '{ROOT_CAUSE}'  (confidence: {RCA_SCORE:.0%})")

### Visualise — Topology Map with RCA Impact Overlay

This dark-mode dependency map colours every service by its topology-adjusted RCA score:
- 🔴 **Red** → Root cause (score > 70%)
- 🟠 **Orange** → High-impact victim (score > 40%)
- 🩷 **Pink** → Affected service (score > 15%)
- 🟢 **Green** → Healthy / not on the critical path

The yellow annotation arrow points at the identified root cause.

In [ ]:
# ── Visualise: Topology Map with RCA Overlay ──────────────────────────────────
# This is the final RCA visualisation: the service dependency graph annotated
# with anomaly data. Nodes are coloured by RCA score — the brightest red node
# is the identified root cause.

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_facecolor("#1a1a2e")
fig.patch.set_facecolor("#1a1a2e")

# Layout — hierarchical from left to right by depth
pos = {
    "mobile-gateway":  (0.0, 0.5),
    "trip-service":    (0.22, 0.75),
    "pricing-service": (0.22, 0.25),
    "payment-service": (0.22, 0.0),
    "driver-service":  (0.50, 0.75),
    "surge-engine":    (0.50, 0.25),
    "fraud-detector":  (0.50, 0.0),
    "billing-db":      (0.78, 0.10),
    "location-db":     (0.78, 0.75),
}

# Node colours by RCA score
rca_score_map = dict(zip(df_rca["service"], df_rca["TOPO_RCA_SCORE"]))
node_colors = []
for node in G.nodes():
    score = rca_score_map.get(node, 0)
    if score > 0.7:
        node_colors.append("#d62728")
    elif score > 0.4:
        node_colors.append("#ff7f0e")
    elif score > 0.15:
        node_colors.append("#e377c2")
    else:
        node_colors.append("#2ca02c")

# Draw edges
nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#aaaaaa",
                       arrows=True, arrowsize=20, width=1.5,
                       connectionstyle="arc3,rad=0.1")

# Draw nodes
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                       node_size=1800, alpha=0.9)

# Labels with error rate
labels = {}
for node in G.nodes():
    err = G.nodes[node].get("peak_err", 0)
    labels[node] = f"{node}\n{err:.0%} err"
nx.draw_networkx_labels(G, pos, labels=labels, ax=ax,
                         font_size=7, font_color="white", font_weight="bold")

# Mark root cause
if ROOT_CAUSE in pos:
    rc_pos = pos[ROOT_CAUSE]
    ax.annotate(f"⭐ ROOT CAUSE\n({RCA_SCORE:.0%} confidence)",
                xy=rc_pos, xytext=(rc_pos[0]+0.1, rc_pos[1]+0.12),
                fontsize=9, color="yellow", fontweight="bold",
                arrowprops=dict(arrowstyle="->", color="yellow"))

# Legend
legend_handles = [
    mpatches.Patch(color="#d62728", label="Root Cause (>70%)"),
    mpatches.Patch(color="#ff7f0e", label="High Impact (>40%)"),
    mpatches.Patch(color="#e377c2", label="Affected (>15%)"),
    mpatches.Patch(color="#2ca02c", label="Healthy / Unaffected"),
]
ax.legend(handles=legend_handles, loc="lower right", facecolor="#2d2d44",
           labelcolor="white", fontsize=9)

ax.set_title("🗺️  Topology-Aware RCA — Service Dependency Map with Impact Overlay",
              color="white", fontsize=13, fontweight="bold", pad=15)
ax.axis("off")

plt.tight_layout()
plt.savefig("/tmp/lab10_topology_rca.png", dpi=150, bbox_inches="tight",
             facecolor="#1a1a2e")
plt.show()
print("✅ Topology RCA map rendered")

---
## Phase 7 — Automated Incident Report 📋

### 📖 Concept: From RCA to Action

> The final output of an AIOps RCA pipeline is not just a cause — it's an **actionable incident report** that an on-call engineer can act on immediately.

A good automated incident report must include:
- **Summary** — What happened, when, and which services are affected
- **Root cause identification** — With evidence and confidence score
- **Impact assessment** — Which services are impacted and how severely
- **Timeline** — When each service was affected
- **Remediation playbook** — Step-by-step fix instructions
- **Prevention recommendations** — How to prevent recurrence

### Remediation Knowledge Base

The `REMEDIATION_KB` maps each known failure type to a structured playbook.
The RCA engine looks up the identified root cause service and retrieves the
correct playbook automatically — no human lookup required.

In production this would be backed by a CMDB, PagerDuty runbooks, or a ServiceNow integration.

In [ ]:
# ── Remediation Knowledge Base ─────────────────────────────────────────────────
# Maps failure patterns to structured playbooks.
# In production this would be backed by a CMDB / runbook system.

REMEDIATION_KB = {
    "location-db": {
        "root_cause_type": "Database OOM Kill",
        "immediate_actions": [
            "kubectl describe pod -n aiops-lab10 -l app=location-db",
            "Increase memory limit: kubectl set resources deployment/location-db -n aiops-lab10 --limits memory=512Mi",
            "Check for memory leaks: kubectl logs -n aiops-lab10 -l app=location-db --previous",
            "If data-loss risk: verify WAL integrity before restart",
        ],
        "recovery_validation": [
            "kubectl wait --for=condition=ready pod -l app=location-db -n aiops-lab10 --timeout=120s",
            "Verify driver-service returns HTTP 200: kubectl exec -n aiops-lab10 deploy/driver-service -- wget -qO- http://location-db:5433/health",
        ],
        "prevention": [
            "Set VPA (Vertical Pod Autoscaler) to auto-resize memory limits",
            "Add PrometheusRule: alert on memory_working_set > 80% of limit",
            "Enable PostgreSQL shared_buffers tuning via ConfigMap",
            "Implement circuit breaker in driver-service for DB connection retries",
        ],
        "runbook_url": "https://wiki.internal/runbooks/database-oom-recovery",
    },
    "driver-service": {
        "root_cause_type": "Downstream DB Unavailable",
        "immediate_actions": ["Check location-db health first", "Review circuit breaker settings"],
        "recovery_validation": ["Verify location-db is healthy", "Check error rate drops"],
        "prevention": ["Add retry with exponential backoff", "Implement read replica fallback"],
        "runbook_url": "https://wiki.internal/runbooks/service-dependency-failure",
    }
}

# ── Generate Incident Report ───────────────────────────────────────────────────
# Compile all RCA evidence into a structured incident report.

now = datetime.now()
incident_id = f"INC-{now.strftime('%Y%m%d')}-{random.randint(1000,9999)}"
playbook = REMEDIATION_KB.get(ROOT_CAUSE, REMEDIATION_KB.get("driver-service"))

affected_table = df_rca[["service", "TOPO_RCA_SCORE", "blast_radius"]].copy()
affected_table.columns = ["Service", "Impact Score", "Blast Radius"]

# Timeline
timeline_rows = sorted([
    (f"T+{v['delay_s']:.0f}s", k, f"{v['peak_err']:.0%} error rate")
    for k, v in first_anomaly.items()
], key=lambda x: float(x[0][2:-1]))

print("=" * 70)
print(f"  🚨 AUTOMATED INCIDENT REPORT")
print("=" * 70)
print(f"  Incident ID   : {incident_id}")
print(f"  Generated     : {now.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Environment   : Kubernetes — aiops-lab10")
print(f"  RCA Engine    : AIOps Session 10 — Topology-Aware RCA")
print("=" * 70)

print("\n📍 EXECUTIVE SUMMARY")
print(f"  A database OOM event in '{ROOT_CAUSE}' caused cascading failures across")
print(f"  {len(INCIDENT_SERVICES)} services in the ride-sharing platform.")
print(f"  Peak user impact: ~{int(first_anomaly.get('mobile-gateway',{}).get('peak_err',0)*100)}% of ride requests failed.")

print(f"\n🔍 ROOT CAUSE")
print(f"  Service        : {ROOT_CAUSE}")
print(f"  Failure type   : {playbook['root_cause_type']}")
print(f"  RCA Confidence : {RCA_SCORE:.0%}")
print(f"  First anomaly  : T+{first_anomaly.get(ROOT_CAUSE,{}).get('delay_s',0):.0f}s")
print(f"  Runbook        : {playbook['runbook_url']}")

print("\n⏱️  PROPAGATION TIMELINE")
print(tabulate(timeline_rows, headers=["Onset", "Service", "Peak Impact"],
               tablefmt="rounded_outline"))

print("\n💥 AFFECTED SERVICES (ranked by impact)")
print(tabulate(affected_table, headers="keys", tablefmt="rounded_outline",
               floatfmt=(".4f", ".0f")))

print("\n🛠️  IMMEDIATE REMEDIATION STEPS")
for i, step in enumerate(playbook["immediate_actions"], 1):
    print(f"  {i}. {step}")

print("\n✅ RECOVERY VALIDATION")
for step in playbook["recovery_validation"]:
    print(f"  → {step}")

print("\n🛡️  PREVENTION RECOMMENDATIONS")
for rec in playbook["prevention"]:
    print(f"  • {rec}")

print("\n" + "=" * 70)
print("  END OF REPORT")
print("=" * 70)

### Final Summary Dashboard

This four-panel dashboard gives a single-view summary of the complete RCA pipeline:

| Panel | Content | Key insight |
|-------|---------|-------------|
| Top-left | Topology RCA score per service | This is the corrected ranking |
| Top-right | Raw peak error rate | What a naive metric alert would show |
| Bottom-left | Evidence heatmap | Which factors drove each service's score |
| Bottom-right | Propagation timeline | Chronological failure onset order |

> 💡 **Compare Panel 1 vs Panel 2.** `mobile-gateway` looks prominent in Panel 2
> (raw error rate). It drops to the bottom in Panel 1 (topology score).
> That difference is topology-aware RCA working correctly.

In [ ]:
# ── Final Summary Dashboard ────────────────────────────────────────────────────
# A four-panel dashboard showing the complete RCA pipeline output:
#   1. RCA Score Ranking (bar chart)
#   2. Error Rate at Peak (bar chart)
#   3. Evidence Heatmap (causal factors per service)
#   4. Propagation Timeline (onset order)

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

# Panel 1: RCA Score Ranking
ax1 = fig.add_subplot(gs[0, 0])
colors_rca = ["#d62728" if s == ROOT_CAUSE else "#5c85d6" for s in df_rca["service"]]
bars = ax1.barh(df_rca["service"], df_rca["TOPO_RCA_SCORE"], color=colors_rca)
ax1.set_xlabel("Topology-Aware RCA Score")
ax1.set_title("🎯 RCA Score Ranking", fontweight="bold")
for bar, val in zip(bars, df_rca["TOPO_RCA_SCORE"]):
    ax1.text(val + 0.01, bar.get_y() + bar.get_height()/2, f"{val:.2f}",
             va="center", fontsize=8)

# Panel 2: Peak Error Rate
ax2 = fig.add_subplot(gs[0, 1])
err_data = df_metrics.groupby("service")["error_rate"].max().sort_values(ascending=False)
colors_err = ["#d62728" if s == ROOT_CAUSE else "#ff9966" for s in err_data.index]
ax2.bar(range(len(err_data)), err_data.values, color=colors_err)
ax2.set_xticks(range(len(err_data)))
ax2.set_xticklabels(err_data.index, rotation=35, ha="right", fontsize=8)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax2.set_title("📊 Peak Error Rate per Service", fontweight="bold")

# Panel 3: Evidence Heatmap
ax3 = fig.add_subplot(gs[1, 0])
heat_data = pd.DataFrame({
    svc: {
        "Peak Err Rate": causal_scores.get(svc, {}).get("peak_err", 0),
        "Temporal Score": 1 - min(causal_scores.get(svc, {}).get("onset_s", 999)/150, 1),
        "K8s Events": min(causal_scores.get(svc, {}).get("k8s_weight", 0)/20, 1),
        "Prec. Score": min(causal_scores.get(svc, {}).get("prec_score", 0)/len(affected), 1),
    }
    for svc in affected
}).T
sns.heatmap(heat_data, ax=ax3, cmap="YlOrRd", annot=True, fmt=".2f",
             linewidths=0.5, cbar_kws={"shrink": 0.8})
ax3.set_title("🔬 Evidence Heatmap per Causal Factor", fontweight="bold")
ax3.tick_params(axis="x", rotation=30)
ax3.tick_params(axis="y", rotation=0, labelsize=8)

# Panel 4: Propagation Timeline
ax4 = fig.add_subplot(gs[1, 1])
timeline_svcs = sorted(first_anomaly.keys(), key=lambda s: first_anomaly[s]["delay_s"])
timeline_onsets = [first_anomaly[s]["delay_s"] for s in timeline_svcs]
colors_t = ["#d62728" if s == ROOT_CAUSE else "#5c85d6" for s in timeline_svcs]
ax4.barh(timeline_svcs, timeline_onsets, color=colors_t)
ax4.set_xlabel("Onset (seconds after T0)")
ax4.set_title("⏱️  Failure Propagation Timeline", fontweight="bold")
ax4.axvline(0, color="red", linestyle="--", alpha=0.5)
for i, (svc, t) in enumerate(zip(timeline_svcs, timeline_onsets)):
    ax4.text(t + 0.5, i, f"T+{t:.0f}s", va="center", fontsize=8)

fig.suptitle(f"🔍 AIOps RCA Dashboard — Incident {incident_id}",
              fontsize=14, fontweight="bold", y=1.02)

plt.savefig("/tmp/lab10_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Summary dashboard rendered")

---
## Phase 8 — Cleanup 🧹

Remove all Kubernetes resources created during this lab.

> ⚠️ This deletes the entire `aiops-lab10` namespace and all its pods, deployments, and services.

In [ ]:
# ── Delete the lab namespace ───────────────────────────────────────────────────
# Deleting the namespace cascades to all pods, deployments, and services within it.
# This is the cleanest way to tear down after a lab.

print(f"Deleting namespace '{NAMESPACE}' and all contained resources...")
out, rc = run(f"kubectl delete namespace {NAMESPACE} --wait=false")

if rc == 0:
    print(f"\n✅ Namespace '{NAMESPACE}' deletion initiated.")
    print("   It may take 30–60 seconds to fully terminate.")
    print("   You can verify with: kubectl get namespace aiops-lab10")
else:
    print(f"\n⚠️  Could not delete namespace — it may not exist or already be deleted.")

---
## 📚 Lab Recap & Key Takeaways

### End-to-End RCA Pipeline Recap

```
 Failure Injection        │
 (OOM at location-db)     │  T0
          │               ▼
 Multi-Source Evidence ───┤  K8s Events + Metrics + Logs
 Collection               │
          │               ▼
 Event Correlation ───────┤  Temporal + Structural + Semantic
          │               │  → Incident boundary identified
          │               ▼
 Causal Analysis ─────────┤  Temporal Precedence + Propagation Tree
          │               │  + Anomaly Scoring
          │               ▼
 Topology-Aware RCA ──────┤  Depth bonus + Blast radius scoring
          │               │  → Root cause: location-db (OOM)
          │               ▼
 Automated Incident  ─────┘  Ranked candidates + Playbook
 Report                       + Prevention recommendations
```

---

### 🏆 Key Concepts Mastered

| Concept | What You Learned |
|---------|------------------|
| **Event Correlation** | Temporal, structural and semantic techniques to group related signals |
| **Causal Analysis** | Temporal precedence, propagation trees, and anomaly scoring to identify causes vs. effects |
| **Topology-Aware RCA** | Using service dependency graphs to re-weight and prune RCA candidates |
| **Blast Radius** | Quantifying how many services would recover from fixing a given candidate |
| **Automated Reporting** | Translating RCA outputs into actionable incident reports with remediation playbooks |

---

### 💡 Why Topology-Aware RCA Wins

> A pure metrics-based system would rank `mobile-gateway` highly (it has a high error rate and serves all user traffic). But the topology graph tells us `mobile-gateway` has 3 hops between it and `location-db` — it's a **downstream victim**, not a cause.
>
> **Topology-awareness routes the engineer directly to the source**, saving minutes or hours of manual triage.

---

### 🔗 Connection to Session Topics

- **Event correlation** → Phase 4 (temporal, structural, semantic)
- **Causal analysis** → Phase 5 (precedence, propagation trees, anomaly scoring)
- **Topology-aware RCA** → Phase 6 (graph traversal, depth bonus, blast radius)

---

*Lab 10 complete. Well done! 🎉*